In [333]:
import pandas as pd
import numpy as np

In [334]:
df=pd.read_csv("Churn_Modelling.csv")

In [335]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [336]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [ ]:
df.drop(columns=["Surname","RowNumber","CustomerId"],inplace=True)

In [338]:
df.shape

(10000, 11)

In [339]:
df.drop_duplicates(inplace=True)

In [340]:
df.shape

(10000, 11)

In [341]:
df["Gender"]=df.Gender.map({"Female":1,"Male":0})

In [342]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,1,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,1,41,1,83807.86,1,0,1,112542.58,0
2,502,France,1,42,8,159660.80,3,1,0,113931.57,1
3,699,France,1,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,1,43,2,125510.82,1,1,1,79084.10,0


In [343]:
df.Geography.unique()

array(['France', 'Spain', 'Germany'], dtype=object)

In [344]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(sparse_output=False,drop='first')

In [345]:
x=df.drop(columns="Exited")
y=df.Exited

In [346]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [347]:
x_train_ohe=ohe.fit_transform(x_train[["Geography"]])
x_test_ohe=ohe.transform(x_test[["Geography"]])

In [348]:
feature=ohe.get_feature_names_out()

In [349]:
x_train_ohe=pd.DataFrame(x_train_ohe,columns=feature)
x_test_ohe=pd.DataFrame(x_test_ohe,columns=feature)

In [350]:
print(x_train_ohe.shape)

(8000, 2)


In [351]:
print(feature)

['Geography_Germany' 'Geography_Spain']


In [352]:
x_train=x_train.reset_index(drop=True)
x_test=x_test.reset_index(drop=True)

In [353]:
x_train=x_train.drop(columns="Geography")
x_test=x_test.drop(columns="Geography")

In [354]:
x_train=pd.concat([x_train,x_train_ohe],axis=1)
x_test=pd.concat([x_test,x_test_ohe],axis=1)

In [355]:
x_train

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain
0,706,1,37,7,0.00,2,1,1,110899.30,0.0,1.0
1,631,1,33,4,123246.70,1,0,0,112687.57,1.0,0.0
2,769,0,34,7,115101.50,1,0,0,57841.89,1.0,0.0
3,605,1,28,6,0.00,2,0,0,159508.52,0.0,0.0
4,438,0,31,8,78398.69,1,1,0,44937.01,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
7995,624,0,44,3,0.00,2,1,0,88407.51,0.0,0.0
7996,692,0,33,9,0.00,1,1,0,113505.93,0.0,0.0
7997,647,0,28,8,0.00,2,1,1,91055.27,0.0,0.0
7998,713,1,25,4,121172.97,1,1,1,56268.98,0.0,0.0


In [356]:
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression()

In [357]:
model=lr.fit(x_train,y_train)

c:\Users\ACER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [358]:
y_pred=model.predict(x_test)

In [359]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7925